# Lesson 2: Character-level tokenization

A tokenizer turns text into the integer IDs a model can process. In this first, deliberately simple tokenizer, one character is one token. The original tutorial puts these ideas into a few compact notebook cells; here we split them into three runnable steps so each idea can be inspected on its own. The C and Java versions use separate source files, so they organize the same steps around vocabulary and lesson classes instead of copying notebook cell boundaries.

Continue after [Lesson 1: read and inspect the training text](lesson_01_dataset.ipynb). The source being followed is [`gpt_dev.ipynb`](../gpt_dev.ipynb), especially the character vocabulary and encode/decode sections. This notebook links to the equivalent [C implementation](../c/src/tokenization_main.c) and [Java implementation](../java/src/main/java/learning/karpathy/gpt1/CharacterTokenizationLesson.java).

## Prepare the shared corpus

This short setup repeats the file-reading idea from Lesson 1 so this lesson can be opened and run directly. It searches the repository root, `gpt-1/`, and this notebook's folder, then reads UTF-8 text as a Python string.

In [ ]:
from pathlib import Path

data_candidates = (
    Path.cwd() / "data" / "input.txt",
    Path.cwd() / "gpt-1" / "data" / "input.txt",
    Path.cwd().parent / "data" / "input.txt",
)
dataset_path = next((path.resolve() for path in data_candidates if path.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError("Could not find gpt-1/data/input.txt. Open the repo root, gpt-1/, or gpt-1/python/.")

text = dataset_path.read_text(encoding="utf-8")
print(f"Dataset: {dataset_path}")
print(f"Text length: {len(text):,} characters")

## Step 1: Build the vocabulary

`set(text)` finds the distinct characters, and `sorted(...)` gives them a stable order. The position of each character in that ordered list becomes its token ID. These IDs are just labels: ID 10 is not inherently larger or more meaningful than ID 2.

The shared Tiny Shakespeare file currently has 65 distinct characters. In C, see `vocabulary_build` in [`vocabulary.c`](../c/src/vocabulary.c); in Java, see `CharacterVocabulary.fromText` in [`CharacterVocabulary.java`](../java/src/main/java/learning/karpathy/gpt1/CharacterVocabulary.java).

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)

print(f"Vocabulary size: {vocab_size}")
print(f"Characters in ID order: {''.join(chars)!r}")

## Step 2: Encode and decode a short example

`stoi` means string-to-integer and `itos` means integer-to-string. `encode` looks up one ID per input character; `decode` performs the reverse lookup and joins the characters back together. This is the same lookup logic used by the `byte_to_id`/`id_to_byte` arrays in C and the maps/lists in Java.

The tutorial next wraps these IDs in a PyTorch tensor. We keep them as ordinary Python integers here so the mapping is visible without adding a tensor dependency; tensors become useful when we create batches for model training.

In [ ]:
stoi = {character: token_id for token_id, character in enumerate(chars)}
itos = {token_id: character for token_id, character in enumerate(chars)}

def encode(sample):
    return [stoi[character] for character in sample]

def decode(token_ids):
    return "".join(itos[token_id] for token_id in token_ids)

sample = "hii there"
sample_ids = encode(sample)
print(f"Input text: {sample!r}")
print(f"Token IDs: {sample_ids}")
print(f"Decoded text: {decode(sample_ids)!r}")
assert decode(sample_ids) == sample

## Step 3: Encode the full corpus

Now the same functions process every character in the training text. Because this is a character tokenizer, the number of token IDs equals the number of characters. The round-trip check verifies that no information was lost. The IDs are still a Python list; the reference notebook converts its list to `torch.tensor` at this point.

In [ ]:
encoded_text = encode(text)
print(f"Token count: {len(encoded_text):,}")
print(f"First 100 token IDs: {encoded_text[:100]}")

round_trip_matches = decode(encoded_text) == text
print(f"Full text round trip matches: {round_trip_matches}")
assert round_trip_matches

## Compare the implementations

Run the corresponding C or Java step from VS Code with **Terminal → Run Task → Run tutorial lesson**, then choose lesson `2`, a language, and a step. Step 3 quietly builds the vocabulary first, just as this notebook relies on the earlier cells having run. The shared runner is documented in [`gpt-1/README.md`](../README.md#run-lessons-by-step).

Try changing the sample to another short phrase from the corpus. Then inspect the vocabulary order and explain why the same text gets the same IDs every time.

For this ASCII corpus, Python's Unicode code points, Java's UTF-16 `char` units, and C's bytes identify the same symbols. That would need more care for arbitrary Unicode text. References: [Python `sorted`](https://docs.python.org/3/library/functions.html#sorted), [Python sets](https://docs.python.org/3/library/stdtypes.html#set), and [PyTorch tensors](https://pytorch.org/docs/stable/tensors.html).